# Modelos finales del prototipo con las otras cuatro semillas

El prototipo muestra, como predicción principal de la LSTM, la de los modelos entrenados con la semilla 42 (`artefactos_lstm/`), la misma que usan los análisis por estudiante del informe (Apéndice C). Este cuaderno entrena el mismo conjunto de 12 modelos (una asignatura por hito) con las otras cuatro semillas del barrido, 7, 123, 2024 y 31, para que la tarjeta de la LSTM pueda mostrar también el rango de la predicción entre semillas.

No cambia ningún resultado del informe ni los modelos de la semilla 42. Como los modelos finales se ajustan con todos los datos, las predicciones de este cuaderno describen el comportamiento del prototipo y no son una evaluación predictiva. Los artefactos se guardan en `artefactos_lstm/semillas_adicionales/<semilla>/`.

Reutiliza `entrenar_modelos_finales.py` (misma arquitectura, mismos hiperparámetros y mismo EarlyStopping), solo cambiando la semilla. Tarda unos minutos por semilla; si una semilla ya está entrenada, se carga su resumen en vez de reentrenar.

In [1]:
import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

AQUI = Path.cwd().resolve()
sys.path.insert(0, str(AQUI))
sys.path.insert(0, str(AQUI / "services"))

import entrenar_modelos_finales as entrenamiento
import lstm_service

logging.getLogger("tensorflow").setLevel(logging.ERROR)  # evita avisos repetidos de TensorFlow al predecir caso por caso
SEMILLAS = list(lstm_service.SEMILLAS_ADICIONALES)
SALIDA = lstm_service.RUTA_SEMILLAS_ADICIONALES
print("Semillas adicionales:", SEMILLAS)
print("Carpeta de salida:", SALIDA)

Semillas adicionales: [7, 123, 2024, 31]
Carpeta de salida: /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/5_prototipo/artefactos_lstm/semillas_adicionales


In [2]:
resumenes = []
for semilla in SEMILLAS:
    carpeta = SALIDA / str(semilla)
    archivo = carpeta / "resumen_entrenamiento.csv"
    if archivo.exists():
        resumen = pd.read_csv(archivo)
        print(f"Semilla {semilla}: ya entrenada, se carga su resumen.")
    else:
        inicio = time.time()
        resumen = entrenamiento.entrenar(carpeta, semilla)
        print(f"Semilla {semilla}: entrenada en {(time.time() - inicio) / 60:.1f} minutos.\n")
    resumen.insert(0, "semilla", semilla)
    resumenes.append(resumen)
resumen_total = pd.concat(resumenes, ignore_index=True)
resumen_total.shape

Semilla 7: ya entrenada, se carga su resumen.
Semilla 123: ya entrenada, se carga su resumen.
Semilla 2024: ya entrenada, se carga su resumen.
Semilla 31: ya entrenada, se carga su resumen.


(48, 9)

In [3]:
# Épocas ejecutadas, mejor época y segundos por modelo; EarlyStopping activo = paró antes del máximo (150).
resumen_total[["semilla", "materia", "hito", "n_registros", "epocas_ejecutadas", "epoca_mejor_val_loss",
               "early_stopping_activo", "segundos"]]

,semilla,materia,hito,n_registros,epocas_ejecutadas,epoca_mejor_val_loss,early_stopping_activo,segundos
0,7,Algoritmos y Programación,4,147,31,11,True,3.1
1,7,Algoritmos y Programación,6,147,21,1,True,2.5
2,7,Algoritmos y Programación,8,147,21,1,True,2.9
3,7,Computación Emergente,4,182,32,12,True,3.1
4,7,Computación Emergente,6,182,32,12,True,3.2
5,7,Computación Emergente,8,182,31,11,True,3.3
6,7,Estructura de Datos,4,108,35,15,True,3.3
7,7,Estructura de Datos,6,108,31,11,True,3.0
8,7,Estructura de Datos,8,108,29,9,True,2.9
9,7,Matemáticas Discretas,4,87,104,84,True,6.0


In [4]:
# Comprobación: cada semilla tiene los 12 modelos y la misma estructura de archivos que la semilla 42.
archivos = {"modelo.keras", "escalador_x.json", "objetivo.json", "imputacion.json"}
for semilla in SEMILLAS:
    carpetas = [c for c in (SALIDA / str(semilla)).iterdir() if c.is_dir()]
    assert len(carpetas) == 12, (semilla, len(carpetas))
    assert all(archivos <= {a.name for a in c.iterdir()} for c in carpetas), semilla
principales = [c for c in lstm_service.RUTA_ARTEFACTOS.iterdir() if c.is_dir() and c.name != "semillas_adicionales"]
assert len(principales) == 12
print("Cada semilla adicional tiene sus 12 modelos con los mismos archivos que los de la semilla 42.")

Cada semilla adicional tiene sus 12 modelos con los mismos archivos que los de la semilla 42.


## Variación de la predicción entre semillas en los 1.572 casos

Para cada registro estudiante-sección y cada hito (S4, S6 y S8) se calcula la predicción de los modelos finales con las cinco semillas y su rango (mayor menos menor). Es lo que verá el usuario en la tarjeta de la LSTM. Como los modelos se ajustaron con todos los datos, es menor que la variación de la validación cruzada del Apéndice C.

In [5]:
datos = lstm_service._preparar_datos()
estatica, X, X_temas = datos["estatica"], datos["X"], datos["X_temas"]
semillas = [lstm_service.SEMILLA_PRINCIPAL] + SEMILLAS
filas = []
for i in range(len(estatica)):
    for hito in lstm_service.HITOS:
        X_hito = lstm_service.truncar_a_hito(X[i:i + 1], hito)
        T_hito = lstm_service.truncar_a_hito(X_temas[i:i + 1], hito)
        acumulado = float(X_hito[:, :, len(lstm_service.ESTATICAS)].sum(axis=1)[0])
        materia = estatica.materia.iloc[i]
        predicciones = {s: lstm_service._predecir_desde_tensor(materia, hito, X_hito, T_hito, acumulado, s) for s in semillas}
        filas.append({"materia": materia, "trimestre": estatica.trimestre.iloc[i], "seccion": estatica.seccion.iloc[i],
                      "estudiante_id": estatica.estudiante_id.iloc[i], "hito": hito,
                      **{f"semilla_{s}": v for s, v in predicciones.items()}})
dispersion = pd.DataFrame(filas)
columnas = [f"semilla_{s}" for s in semillas]
dispersion["minimo"] = dispersion[columnas].min(axis=1)
dispersion["maximo"] = dispersion[columnas].max(axis=1)
dispersion["rango"] = dispersion.maximo - dispersion.minimo
dispersion.to_csv(SALIDA / "predicciones_prototipo_cinco_semillas.csv", index=False)
len(dispersion)

1572

In [6]:
print("Rango entre semillas de la predicción (participaciones), todos los casos:")
print(dispersion.rango.describe(percentiles=[.5, .9, .95]).round(2).to_string())
print("\nPor asignatura:")
print(dispersion.groupby("materia").rango.agg(["median", "mean", "max"]).round(2))
print("\nPor hito:")
print(dispersion.groupby("hito").rango.agg(["median", "mean", "max"]).round(2))
print("\nProporción de casos con rango mayor que 1 y que 2 participaciones:",
      round((dispersion.rango > 1).mean(), 3), round((dispersion.rango > 2).mean(), 3))

Rango entre semillas de la predicción (participaciones), todos los casos:
count    1572.00
mean        0.69
std         0.45
min         0.06
50%         0.58
90%         1.22
95%         1.52
max         3.36

Por asignatura:
                           median  mean   max
materia                                      
Algoritmos y Programación    0.60  0.71  2.33
Computación Emergente        0.46  0.57  2.92
Estructura de Datos          0.78  0.85  3.19
Matemáticas Discretas        0.50  0.67  3.36

Por hito:
      median  mean   max
hito                    
4       0.68  0.80  3.27
6       0.69  0.76  3.36
8       0.40  0.50  2.92

Proporción de casos con rango mayor que 1 y que 2 participaciones: 0.186 0.022


In [7]:
# La predicción principal del prototipo (semilla 42) siempre queda dentro del rango mostrado.
assert ((dispersion.semilla_42 >= dispersion.minimo - 1e-9) & (dispersion.semilla_42 <= dispersion.maximo + 1e-9)).all()
dispersion[["materia", "trimestre", "seccion", "estudiante_id", "hito", "semilla_42", "minimo", "maximo", "rango"]].head(10)

,materia,trimestre,seccion,estudiante_id,hito,semilla_42,minimo,maximo,rango
0,Algoritmos y Programación,2425-2,1,anon_001,4,4.864684,3.878585,4.889596,1.011010
1,Algoritmos y Programación,2425-2,1,anon_001,6,3.694697,3.231448,4.427576,1.196128
2,Algoritmos y Programación,2425-2,1,anon_001,8,2.743788,2.564372,3.305481,0.741110
3,Algoritmos y Programación,2425-2,1,anon_002,4,1.989459,1.353001,2.360292,1.007291
4,Algoritmos y Programación,2425-2,1,anon_002,6,1.525422,1.163500,2.445844,1.282343
5,Algoritmos y Programación,2425-2,1,anon_002,8,0.678652,0.546303,1.322265,0.775962
6,Algoritmos y Programación,2425-2,1,anon_003,4,9.684691,8.721904,9.684691,0.962787
7,Algoritmos y Programación,2425-2,1,anon_003,6,11.108501,10.686335,11.620027,0.933692
8,Algoritmos y Programación,2425-2,1,anon_003,8,16.061197,15.724914,16.272268,0.547354
9,Algoritmos y Programación,2425-2,1,anon_004,4,4.848995,4.000988,4.848995,0.848008
